In [1]:
# Cell 1: Import libraries and load raw data
import pandas as pd
import numpy as np
import re
import os

# Load the raw data we just scraped
df = pd.read_csv('data/raw_reviews.csv')

print("✓ Data loaded!")
print("Shape:", df.shape)
print("\nFirst look at the data:")
df.head(3)

✓ Data loaded!
Shape: (1200, 12)

First look at the data:


,reviewId,userName,userImage,content,score,thumbsUpCount,reviewCreatedVersion,at,replyContent,repliedAt,appVersion,app_name
0,dcec65b7-5e84-46a3-b4d5-c4f0cc1a85b2,Tkash,https://play-lh.googleusercontent.com/a/ACg8oc...,Hello Habitika! You are the best! I want to he...,5,0,4.9.7,2026-07-14 18:40:22,NaN,NaN,4.9.7,Habitica
1,83c91808-bde0-43fa-b516-ba0e2b74cad3,زهرا بابایی,https://play-lh.googleusercontent.com/a/ACg8oc...,How can I silence Justin in the app? He won't ...,3,0,4.9.7,2026-07-14 12:40:59,NaN,NaN,4.9.7,Habitica
2,ee7c5d85-8a15-41e6-9e7b-071efe8dcdca,Mark,https://play-lh.googleusercontent.com/a-/ALV-U...,Good idea but there aren't enough customizable...,4,0,NaN,2026-07-14 11:54:13,NaN,NaN,NaN,Habitica


In [2]:
# Cell 2: Understand what we have before cleaning anything

print("=== DATA TYPES ===")
print(df.dtypes)

print("\n=== MISSING VALUES ===")
print(df.isnull().sum())

print("\n=== RATING DISTRIBUTION ===")
print(df['score'].value_counts().sort_index())

print("\n=== REVIEWS PER APP ===")
print(df['app_name'].value_counts())

=== DATA TYPES ===
reviewId                  str
userName                  str
userImage                 str
content                   str
score                   int64
thumbsUpCount           int64
reviewCreatedVersion      str
at                        str
replyContent              str
repliedAt                 str
appVersion                str
app_name                  str
dtype: object

=== MISSING VALUES ===
reviewId                  0
userName                  0
userImage                 0
content                   0
score                     0
thumbsUpCount             0
reviewCreatedVersion    224
at                        0
replyContent            310
repliedAt               310
appVersion              224
app_name                  0
dtype: int64

=== RATING DISTRIBUTION ===
score
1    451
2    131
3    102
4    112
5    404
Name: count, dtype: int64

=== REVIEWS PER APP ===
app_name
Habitica        300
Headspace       300
MyFitnessPal    300
Fabulous        300
Name: count, d

In [3]:
# Cell 3: Select only columns we need
# We dont need userImage, replyContent etc.

df = df[[
    'app_name',
    'userName',
    'content',
    'score',
    'thumbsUpCount',
    'at',
    'appVersion'
]]

# Rename columns to cleaner names
df.columns = [
    'app_name',
    'user_name',
    'review_text',
    'rating',
    'helpful_count',
    'review_date',
    'app_version'
]

print("✓ Columns selected and renamed!")
print("New columns:", df.columns.tolist())
print("Shape:", df.shape)


✓ Columns selected and renamed!
New columns: ['app_name', 'user_name', 'review_text', 'rating', 'helpful_count', 'review_date', 'app_version']
Shape: (1200, 7)


In [4]:
# Cell 4: Fix missing values

print("Missing values BEFORE:")
print(df.isnull().sum())

# review_text: drop rows where review is empty
# (we cant analyze sentiment of nothing)
df = df.dropna(subset=['review_text'])

# rating: drop rows where rating is missing
# (its our most important column)
df = df.dropna(subset=['rating'])

# helpful_count: fill missing with 0
df['helpful_count'] = df['helpful_count'].fillna(0)

# app_version: fill missing with Unknown
df['app_version'] = df['app_version'].fillna('Unknown')

print("\nMissing values AFTER:")
print(df.isnull().sum())
print("\nRows remaining:", len(df))

Missing values BEFORE:
app_name           0
user_name          0
review_text        0
rating             0
helpful_count      0
review_date        0
app_version      224
dtype: int64

Missing values AFTER:
app_name         0
user_name        0
review_text      0
rating           0
helpful_count    0
review_date      0
app_version      0
dtype: int64

Rows remaining: 1200


In [5]:
# Cell 5: Remove duplicate reviews

print("Rows BEFORE removing duplicates:", len(df))

df = df.drop_duplicates(
    subset=['app_name', 'user_name', 'review_text']
)

print("Rows AFTER removing duplicates :", len(df))
print("Duplicates removed             :", 1200 - len(df))

Rows BEFORE removing duplicates: 1200
Rows AFTER removing duplicates : 1200
Duplicates removed             : 0


In [6]:
# Cell 6: Convert date column to proper datetime format

# Right now review_date is just text like "2024-03-15 10:30:00"
# We need to convert it to a real date object

df['review_date'] = pd.to_datetime(df['review_date'])

# Extract useful parts from the date
df['review_year']  = df['review_date'].dt.year
df['review_month'] = df['review_date'].dt.month

print("✓ Dates fixed!")
print("\nDate range of reviews:")
print("Earliest:", df['review_date'].min())
print("Latest  :", df['review_date'].max())
print("\nReviews per year:")
print(df['review_year'].value_counts().sort_index())

✓ Dates fixed!

Date range of reviews:
Earliest: 2026-01-10 14:36:27
Latest  : 2026-07-16 12:41:17

Reviews per year:
review_year
2026    1200
Name: count, dtype: int64


In [7]:
# Cell 7: Clean the actual review text

def clean_text(text):
    """
    Takes a messy review → returns clean text
    
    We do this so our sentiment tool can read it properly
    """
    
    # If not text, return empty string
    if not isinstance(text, str):
        return ''
    
    # Make everything lowercase
    text = text.lower()
    
    # Remove website links
    text = re.sub(r'http\S+|www\S+', '', text)
    
    # Remove emojis (keep only normal characters)
    text = text.encode('ascii', 'ignore').decode('ascii')
    
    # Remove special characters
    # Keep only: letters, numbers, spaces, . , ! ? ' "
    text = re.sub(r'[^a-zA-Z0-9\s\.,!?\'"-]', '', text)
    
    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text


# Apply cleaning to every review
df['review_text_clean'] = df['review_text'].apply(clean_text)

# Count words in each review
df['review_word_count'] = df['review_text_clean'].apply(
    lambda x: len(x.split())
)

# Remove reviews that are too short after cleaning
# (less than 3 words = not useful)
df = df[df['review_word_count'] >= 3]

print("✓ Text cleaned!")
print("\nSample original text:")
print(df['review_text'].iloc[0])
print("\nSame text after cleaning:")
print(df['review_text_clean'].iloc[0])
print("\nAverage words per review:", df['review_word_count'].mean().round(1))

✓ Text cleaned!

Sample original text:
Hello Habitika! You are the best! I want to help you with ideas! Can you tell me where I can share an idea? I just like everything in your game, but I didn't find the accessories I wanted for my character. But don't be offended, I'm just trying to make your game diverse so that everyone can find what they want!

Same text after cleaning:
hello habitika! you are the best! i want to help you with ideas! can you tell me where i can share an idea? i just like everything in your game, but i didn't find the accessories i wanted for my character. but don't be offended, i'm just trying to make your game diverse so that everyone can find what they want!

Average words per review: 33.9


In [8]:
# Cell 8: Save the cleaned data

os.makedirs('data', exist_ok=True)
df.to_csv('data/clean_reviews.csv', index=False)

print("✓ Clean data saved!")
print("\n=== CLEANING SUMMARY ===")
print(f"Total reviews       : {len(df)}")
print(f"Apps included       : {df['app_name'].unique()}")
print(f"Date range          : {df['review_date'].min().date()} "
      f"to {df['review_date'].max().date()}")
print(f"Average rating      : {df['rating'].mean():.2f} stars")
print(f"Average review len  : {df['review_word_count'].mean():.0f} words")
print(f"\nRating breakdown:")
print(df['rating'].value_counts().sort_index())

✓ Clean data saved!

=== CLEANING SUMMARY ===
Total reviews       : 1111
Apps included       : <StringArray>
['Habitica', 'Headspace', 'MyFitnessPal', 'Fabulous']
Length: 4, dtype: str
Date range          : 2026-01-10 to 2026-07-16
Average rating      : 2.79 stars
Average review len  : 34 words

Rating breakdown:
rating
1    438
2    130
3     99
4    111
5    333
Name: count, dtype: int64
